In [1]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser



/Users/raj/Documents/personal/ai_trading_system/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
from pathlib import Path
current_file = Path('/Users/raj/Documents/personal/ai_trading_system/finance_agent/src/hf_langchain_explore.ipynb').resolve()
parent_dir = current_file.parent.parent.parent
print(f"parent dir = {parent_dir}")
sys.path.append(str(parent_dir))

from finance_agent.src.lib.tool_functions import search_fundamentals

parent dir = /Users/raj/Documents/personal/ai_trading_system


In [3]:
print(callable(search_fundamentals))
print(repr(search_fundamentals))
print(type(search_fundamentals))

from langchain_core.utils.function_calling import convert_to_openai_tool

print(convert_to_openai_tool(search_fundamentals))

False
StructuredTool(name='search_fundamentals', description='function to get EBITDA values for a company', args_schema=<class 'langchain_core.utils.pydantic.search_fundamentals'>, func=<function search_fundamentals at 0x177e7ab60>)
<class 'langchain_core.tools.structured.StructuredTool'>
{'type': 'function', 'function': {'name': 'search_fundamentals', 'description': 'function to get EBITDA values for a company', 'parameters': {'properties': {'question': {'type': 'string'}}, 'required': ['question'], 'type': 'object'}}}


In [6]:
hf_model = pipeline(
    'text-generation',
    model="Qwen/Qwen2.5-7B-Instruct"
)

Loading weights: 100%|██████████| 339/339 [00:00<00:00, 13574.06it/s]


In [7]:
print(search_fundamentals.args_schema.model_json_schema())

{'description': 'function to get EBITDA values for a company', 'properties': {'question': {'title': 'Question', 'type': 'string'}}, 'required': ['question'], 'title': 'search_fundamentals', 'type': 'object'}


In [8]:
llm = HuggingFacePipeline(pipeline=hf_model)

llm.invoke("Tell me a joke")

tools = [search_fundamentals]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [9]:
callable(search_fundamentals)

False

In [10]:
chat_model = ChatHuggingFace(llm=llm)

model_with_tools = chat_model.bind_tools(tools)



In [25]:
from langchain_core.messages import HumanMessage

messages = [
    HumanMessage(
        content="What is Reliance Industries' EBITDA?"
    )
]

prompt = chat_model._to_chat_prompt(messages)

print(prompt)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is Reliance Industries' EBITDA?<|im_end|>
<|im_start|>assistant



In [ ]:
prompt = PromptTemplate.from_template(
    "You are a financial expert can you look into this question: {question}"
)

chain = prompt | llm | StrOutputParser()

result = chain.invoke({'question': 'what is the revenue of reliance company this year!'})

print(result)

In [11]:
print(type(model_with_tools))
print(model_with_tools)

<class 'langchain_core.language_models.chat_models._ChatModelBinding'>
bound=ChatHuggingFace(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-huggingface': '1.2.3'}}, llm=HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-huggingface': '1.2.3'}}, pipeline=TextGenerationPipeline: {'model': 'Qwen2ForCausalLM', 'dtype': 'bfloat16', 'device': 'mps', 'input_modalities': 'text', 'output_modalities': ('text',)}, model_id='Qwen/Qwen2.5-7B-Instruct'), tokenizer=Qwen2Tokenizer(name_or_path='Qwen/Qwen2.5-7B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, norma

In [20]:
tool_schema = {
    "name": search_fundamentals.name,
    "description": search_fundamentals.description,
    "parameters": search_fundamentals.args_schema.model_json_schema(),
}

print(tool_schema)

model_with_tools = chat_model.bind_tools([tool_schema])

{'name': 'search_fundamentals', 'description': 'function to get EBITDA values for a company', 'parameters': {'description': 'function to get EBITDA values for a company', 'properties': {'question': {'title': 'Question', 'type': 'string'}}, 'required': ['question'], 'title': 'search_fundamentals', 'type': 'object'}}


In [51]:
resp = model_with_tools.invoke("what i the revenu of reliance?")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [52]:
resp.content

"<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nwhat i the revenu of reliance?<|im_end|>\n<|im_start|>assistant\nReliance Industries Limited (RIL) is one of India's largest and most diversified companies, with revenues that can vary significantly from year to year due to fluctuations in global oil prices, commodity markets, and other economic factors. To get the most accurate and up-to-date revenue figures for Reliance Industries, you would need to refer to their latest financial reports or press releases.\n\nFor the most recent financial year data, you can check:\n\n1. **Annual Report**: Reliance Industries typically publishes an annual report which includes detailed financial information.\n2. **Quarterly Reports**: These provide interim financial updates and can be found on the company's official website or stock exchange filings.\n3. **Stock Exchange Filings**: The Bombay Stock Exchange (BSE) and National Stock E

In [23]:
print("BOUND RUNNABLE:")
print(model_with_tools.bound)

print("BOUND KWARGS:")
print(model_with_tools.kwargs)

BOUND RUNNABLE:
metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-huggingface': '1.2.3'}} llm=HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-huggingface': '1.2.3'}}, pipeline=TextGenerationPipeline: {'model': 'Qwen2ForCausalLM', 'dtype': 'bfloat16', 'device': 'mps', 'input_modalities': 'text', 'output_modalities': ('text',)}, model_id='Qwen/Qwen2.5-7B-Instruct') tokenizer=Qwen2Tokenizer(name_or_path='Qwen/Qwen2.5-7B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lst

In [29]:
from langchain_core.language_models.chat_models import _ChatModelBinding

print(inspect.getsource(_ChatModelBinding))

class _ChatModelBinding(RunnableBinding[LanguageModelInput, AIMessage]):  # type: ignore[no-redef]
    """`RunnableBinding` that preserves chat-model-typed v3 overloads.

    Returned by `BaseChatModel.bind` so that callers of the bound runnable's
    `stream_events(version="v3")` / `astream_events(version="v3")` get the
    typed `ChatModelStream` / `AsyncChatModelStream` back without needing
    `cast`. At runtime this is a plain `RunnableBinding`; the subclass
    exists purely to give the type checker a more specific surface.

    The chat-model narrowing is preserved across further `bind` /
    `with_config` calls because `RunnableBinding.bind` constructs its
    result via `self.__class__(...)`.
    """

    @classmethod
    @override
    def lc_id(cls) -> list[str]:
        """Serialize as `RunnableBinding`.

        At runtime this class is behaviorally identical to `RunnableBinding`;
        keeping the serialized id stable means existing snapshots and the
        load mapping

In [ ]:
import inspect

print(inspect.getsource(ChatHuggingFace.bind_tools))
print(inspect.getsource(ChatHuggingFace._to_chat_prompt))
print(inspect.getsource(ChatHuggingFace._to_chatml_format))

    def bind_tools(
        self,
        tools: Sequence[dict[str, Any] | type | Callable | BaseTool],
        *,
        tool_choice: dict | str | bool | None = None,
        **kwargs: Any,
    ) -> Runnable[LanguageModelInput, AIMessage]:
        """Bind tool-like objects to this chat model.

        Assumes model is compatible with OpenAI tool-calling API.

        Args:
            tools: A list of tool definitions to bind to this chat model.

                Supports any tool definition handled by [`convert_to_openai_tool`][langchain_core.utils.function_calling.convert_to_openai_tool].
            tool_choice: Which tool to require the model to call.
                Must be the name of the single provided function or
                `'auto'` to automatically determine which function to call
                (if any), or a dict of the form:
                {"type": "function", "function": {"name": <<tool_name>>}}.
            **kwargs: Any additional parameters to pass to the
     

In [31]:
print(model_with_tools.kwargs)

{'tools': [{'type': 'function', 'function': {'name': 'search_fundamentals', 'description': 'function to get EBITDA values for a company', 'parameters': {'description': 'function to get EBITDA values for a company', 'properties': {'question': {'title': 'Question', 'type': 'string'}}, 'required': ['question'], 'title': 'search_fundamentals', 'type': 'object'}}}]}


In [30]:
print(chat_model.tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [22]:
from langchain_core.messages import HumanMessage, SystemMessage

response = model_with_tools.invoke([
    SystemMessage(content=f"You are financial expert and you use tools to answer user questions using tool call schema {tool_schema}"),
    HumanMessage(content="What is Reliance Industries' EBITDA this year?")
])

print(dir(response))
print(response.content)
print(response.tool_calls)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


['__abstractmethods__', '__add__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_extra_info__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__pydantic_private__', '__pydantic_root_model__', '__pydantic_serializer__', '__pyda

In [ ]:
from langchain_core.messages import HumanMessage

messages = [
    HumanMessage(
        content="What is Reliance Industries' EBITDA?"
    )
]

prompt = chat_model._to_chat_prompt(messages)

print(prompt)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is Reliance Industries' EBITDA?<|im_end|>
<|im_start|>assistant



In [41]:
from langchain_core.messages import HumanMessage

messages = [
    {
        "role": "user",
        "content": "What is Reliance Industries' EBITDA?"
    }
]

tool_definitions = model_with_tools.kwargs["tools"]

print(f"tool definition = {tool_definitions}")

prompt = chat_model.tokenizer.apply_chat_template(
    messages,
    tools=tool_definitions,
    tokenize=False,
    add_generation_prompt=True,
)

print(prompt)

tool definition = [{'type': 'function', 'function': {'name': 'search_fundamentals', 'description': 'function to get EBITDA values for a company', 'parameters': {'description': 'function to get EBITDA values for a company', 'properties': {'question': {'title': 'Question', 'type': 'string'}}, 'required': ['question'], 'title': 'search_fundamentals', 'type': 'object'}}}]
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "search_fundamentals", "description": "function to get EBITDA values for a company", "parameters": {"description": "function to get EBITDA values for a company", "properties": {"question": {"title": "Question", "type": "string"}}, "required": ["question"], "title": "search_fundamentals", "type": "object"}}}
</tools>

For each function ca

In [ ]:
response = llm.invoke(prompt)
print(response)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "search_fundamentals", "description": "function to get EBITDA values for a company", "parameters": {"description": "function to get EBITDA values for a company", "properties": {"question": {"title": "Question", "type": "string"}}, "required": ["question"], "title": "search_fundamentals", "type": "object"}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
What is Reliance Industries' EBITDA?<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "search_fundamentals", "arguments": {"question": "What is Reliance Indu

In [43]:
resp = model_with_tools.invoke(prompt)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [46]:
print(resp.content)
print(resp.tool_calls)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "search_fundamentals", "description": "function to get EBITDA values for a company", "parameters": {"description": "function to get EBITDA values for a company", "properties": {"question": {"title": "Question", "type": "string"}}, "required": ["question"], "title": "search_fundamentals", "type": "object"}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
What is Reliance Industries' EBITDA?<|im_end|>

In [47]:
model_with_tools = chat_model.bind_tools(tools)

response = model_with_tools.invoke(
    "What is Reliance Industries' EBITDA?"
)

print(response.tool_calls)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[]


In [48]:
print(type(resp))
print("Content type:", type(resp.content))
print("Tool calls:", resp.tool_calls)
print("Invalid tool calls:", resp.invalid_tool_calls)
print("Usage metadata:", resp.usage_metadata)

<class 'langchain_core.messages.ai.AIMessage'>
Content type: <class 'str'>
Tool calls: []
Invalid tool calls: []
Usage metadata: None


In [50]:
llm = HuggingFacePipeline(
    pipeline=hf_model,
    pipeline_kwargs={"return_full_text": False},
)

chat_model = ChatHuggingFace(llm=llm)

model_with_tools = chat_model.bind_tools(tools)

response = model_with_tools.invoke(
    "What is Reliance Industries' EBITDA?"
)

print(response.content)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is Reliance Industries' EBITDA?<|im_end|>
<|im_start|>assistant
Reliance Industries Limited (RIL) is one of India's largest and most diversified companies, with operations spanning oil and gas, petrochemicals, telecommunications, retail, and textiles. EBITDA (Earnings Before Interest, Taxes, Depreciation, and Amortization) is a commonly used financial metric that provides insight into the operating performance of a company.

As of my last update in October 2023, I don't have the exact EBITDA figure for Reliance Industries for the latest fiscal year. To get the most accurate and up-to-date information on RIL's EBITDA, you should refer to their latest financial reports or press releases, which can be found on their official website or financial news platforms like Bloomberg, Reuters, or Yahoo Finance.

For the most recent figures, I recommend checking the following sour

In [ ]:
chat_model.invoke

In [53]:

import json
import re
import uuid
from typing import Any

from pydantic import ConfigDict, Field

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import (
    AIMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool


class QwenToolChatModel(BaseChatModel):
    """Connect a local Hugging Face text-generation pipeline to LangChain tools."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    pipeline: Any
    tokenizer: Any
    generation_kwargs: dict[str, Any] = Field(
        default_factory=lambda: {"max_new_tokens": 256, "do_sample": False}
    )

    @property
    def _llm_type(self) -> str:
        return "qwen-local-tool-chat"

    def bind_tools(self, tools, **kwargs):
        """Convert LangChain tools into schemas understood by Qwen."""
        formatted_tools = [
            convert_to_openai_tool(tool) for tool in tools
        ]
        return self.bind(tools=formatted_tools, **kwargs)

    def _convert_messages(self, messages: list[BaseMessage]) -> list[dict]:
        """Convert LangChain messages to the format expected by Qwen's template."""
        result = []

        for message in messages:
            if isinstance(message, SystemMessage):
                result.append({
                    "role": "system",
                    "content": message.content,
                })

            elif isinstance(message, HumanMessage):
                result.append({
                    "role": "user",
                    "content": message.content,
                })

            elif isinstance(message, ToolMessage):
                result.append({
                    "role": "tool",
                    "name": message.name or "tool",
                    "content": message.content,
                    "tool_call_id": message.tool_call_id,
                })

            elif isinstance(message, AIMessage):
                item = {
                    "role": "assistant",
                    "content": message.content or "",
                }

                if message.tool_calls:
                    item["tool_calls"] = [
                        {
                            "type": "function",
                            "function": {
                                "name": call["name"],
                                "arguments": call["args"],
                            },
                            "id": call.get("id"),
                        }
                        for call in message.tool_calls
                    ]

                result.append(item)

            else:
                raise TypeError(
                    f"Unsupported message type: {type(message).__name__}"
                )

        return result

    def _parse_response(self, text: str) -> AIMessage:
        """Convert Qwen's tool-call markup into LangChain tool calls."""
        matches = list(
            re.finditer(
                r"<tool_call>\s*(.*?)\s*</tool_call>",
                text,
                flags=re.DOTALL,
            )
        )

        if not matches:
            return AIMessage(content=text.strip())

        tool_calls = []
        previous_end = 0
        text_parts = []

        for match in matches:
            # Preserve any ordinary assistant text before a tool call.
            prefix = text[previous_end:match.start()].strip()
            if prefix:
                text_parts.append(prefix)

            try:
                payload = json.loads(match.group(1))
                name = payload["name"]
                args = payload["arguments"]

                # Some models return arguments as a JSON string.
                if isinstance(args, str):
                    args = json.loads(args)

                if not isinstance(args, dict):
                    raise ValueError("Tool arguments must be a JSON object")

                tool_calls.append({
                    "name": name,
                    "args": args,
                    "id": f"call_{uuid.uuid4().hex[:12]}",
                    "type": "tool_call",
                })

            except (json.JSONDecodeError, KeyError, TypeError, ValueError):
                # Do not silently turn malformed tool markup into a tool call.
                text_parts.append(match.group(0))

            previous_end = match.end()

        suffix = text[previous_end:].strip()
        if suffix:
            text_parts.append(suffix)

        return AIMessage(
            content="\n".join(text_parts),
            tool_calls=tool_calls,
        )

    def _generate(
        self,
        messages: list[BaseMessage],
        stop: list[str] | None = None,
        run_manager=None,
        **kwargs: Any,
    ) -> ChatResult:
        """Build the prompt, run Qwen, and return a LangChain message."""
        tools = kwargs.pop("tools", None)

        hf_messages = self._convert_messages(messages)

        prompt = self.tokenizer.apply_chat_template(
            hf_messages,
            tools=tools,
            tokenize=False,
            add_generation_prompt=True,
        )

        generation_kwargs = {
            **self.generation_kwargs,
            **kwargs,
            "return_full_text": False,
        }

        output = self.pipeline(prompt, **generation_kwargs)

        # TextGenerationPipeline normally returns a list of dictionaries.
        if isinstance(output, list) and output:
            generated_text = output[0]["generated_text"]
        elif isinstance(output, dict):
            generated_text = output["generated_text"]
        else:
            raise ValueError(f"Unexpected pipeline output: {output!r}")

        # Apply stop sequences if supplied.
        if stop:
            stop_positions = [
                generated_text.find(token)
                for token in stop
                if token and token in generated_text
            ]
            if stop_positions:
                generated_text = generated_text[:min(stop_positions)]

        message = self._parse_response(generated_text)

        return ChatResult(
            generations=[ChatGeneration(message=message)]
        )


In [56]:
# import QwenToolChatModel
from finance_agent.src.lib.tool_functions import search_fundamentals

qwen_chat = QwenToolChatModel(
    pipeline=llm.pipeline,
    tokenizer=chat_model.tokenizer,
    generation_kwargs={
        "max_new_tokens": 256,
        "do_sample": False,
    },
)

tools = [search_fundamentals]

model_with_tools = qwen_chat.bind_tools(tools)

In [57]:
from langchain_core.messages import HumanMessage

response = model_with_tools.invoke([
    HumanMessage(
        content="What is Reliance Industries' EBITDA?"
    )
])

print("Content:", response.content)
print("Tool calls:", response.tool_calls)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Content: 
Tool calls: [{'name': 'search_fundamentals', 'args': {'question': 'Reliance Industries EBITDA'}, 'id': 'call_bb21b7432161', 'type': 'tool_call'}]
